# Importing the libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Importing the dataset

In [ ]:
dataset = pd.read_csv('Breast_cancer_data.csv')


In [ ]:
dataset

In [ ]:
dataset.head()

In [ ]:
dataset.shape

# Checking Null values

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset.info()

# Correlation

In [ ]:
correlation = dataset.corr()

# Construction Heatmap

In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(correlation,cbar=True,square=True,fmt='.1f',annot_kws={'size':8},cmap='Blues')

# Spltting the dataset into Dependent part and Independent part

In [ ]:
feature_columns = [
    'mean_radius',
    'mean_texture',
    'mean_perimeter',
    'mean_area',
    'mean_smoothness',
]
required_columns = feature_columns + ['diagnosis']
missing_columns = set(required_columns) - set(dataset.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")
if dataset['diagnosis'].nunique() < 2:
    raise ValueError('The diagnosis column must contain at least two classes.')

X = dataset[feature_columns].to_numpy()
y = dataset['diagnosis'].to_numpy()

# Splitting the dataset into the Training set and Test set

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

# Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Training the Random Forest Classification model on the Training set

In [ ]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
classifier.fit(X_train, y_train)

# Predicting the Test set results

In [ ]:
y_pred = classifier.predict(X_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

# Making the Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
y_pred = classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test, y_pred, zero_division=0))
accuracy_score(y_test, y_pred)

In [ ]:
plt.scatter(y_test, y_pred)
plt.xlabel("Actual data")
plt.ylabel("Predicted data")
plt.title("Actual data vs Predicted data")
plt.show()

# Predicting the new result

In [ ]:
# Demonstrate a prediction on an existing held-out row; this is not a diagnosis.
predicted_class = classifier.predict(X_test[[0]])[0]
print(f"Predicted model class for the first held-out sample: {predicted_class}")

# Saving the trained model

In [ ]:
import pickle

In [ ]:
with open('breastcancer.pkl', 'wb') as model_file:
    pickle.dump(classifier, model_file)

In [ ]:
with open('sc.pkl', 'wb') as scaler_file:
    pickle.dump(sc, scaler_file)